# LifeMate: đo thời gian từng mẫu

Notebook chạy **đúng hệ thống của `/code/predict.sh`**: PhoWhisper-large fine-tune (ASR) → Qwen3.5-4B-text-only NF4 + LoRA gộp hai seed (agent gọi tool, bật search guard) → `zaic_tools.ToolEnv`, cùng cấu hình và seed. Khác `predict.sh` ở chỗ chạy **từng mẫu một**: đọc wav → ASR → agent → tool.

- Chỉ cần **Run All**. Kết quả: `/result/jupyter_submission.jsonl`, cùng định dạng `submission.jsonl` và có trường `time`.
- `time` (mili giây) tính từ trước khi đọc audio của mẫu tới khi có `tool_calls` và `state_hash`; không tính thời gian nạp model và warm-up.
- Mỗi mẫu đúng một `ToolEnv`; `zaic_tools` nạp từ `/data/<split>/`.

In [ ]:
# Đường dẫn và cấu hình. Mọi giá trị mặc định là bản nộp; biến môi trường chỉ dùng khi thử ngoài máy chấm
# (ví dụ trên laptop: DATA_ROOT=../data/train+public_test SPLIT=public_test RESULT_DIR=runs/nb_pub).
import json
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")  # chấm không có mạng: mọi model nằm sẵn trong image
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.dont_write_bytecode = True  # không ghi __pycache__ vào zaic_tools trong /data (chỉ đọc)


def find_lifemate_dir() -> Path:
    """Thư mục chứa lifemate/, scripts/, ckpt/: LIFEMATE_DIR nếu có, nếu không thì thư mục của notebook,
    thư mục cha, /code, /app/lifemate (image 05/11)."""
    if os.environ.get("LIFEMATE_DIR"):
        return Path(os.environ["LIFEMATE_DIR"]).resolve()
    for d in (Path.cwd(), Path.cwd().parent, Path("/code"), Path("/app/lifemate")):
        if (d / "lifemate" / "infer.py").is_file():
            return d.resolve()
    raise FileNotFoundError("không tìm thấy code lifemate: đặt LIFEMATE_DIR")


LIFEMATE_DIR = find_lifemate_dir()
sys.path.insert(0, str(LIFEMATE_DIR))
from lifemate.submit import find_split, model_path, settings  # noqa: E402

CFG = settings()  # DEFAULTS của lifemate/submit.py, giống predict.sh
DATA_ROOT = Path(os.environ.get("DATA_ROOT") or "/data").resolve()
RESULT_DIR = Path(os.environ.get("RESULT_DIR") or "/result")
print("code:", LIFEMATE_DIR, "| python:", sys.executable)
print("cấu hình:", CFG)

## Bước 1: seed cố định

In [ ]:
# Seed 0, cùng seed với predict.sh và lúc train. Giải mã greedy nên kết quả không phụ thuộc seed;
# seed chỉ cố định những gì ngẫu nhiên ngoài ý muốn.
import torch

from lifemate.seeding import seed_everything  # random, PYTHONHASHSEED, numpy, torch, cuda, cudnn như mẫu của BTC

seed_everything(CFG["seed"])
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có")

## Bước 2: nạp model và tài nguyên

In [ ]:
# Không dùng API ngoài: ASR, LLM và tokenizer nằm sẵn trong image. Không tính vào thời gian của mẫu nào.
from lifemate.agent import HFBackend
from lifemate.asr import Whisper
from lifemate.submit import MAX_TOKENS

t_load = time.perf_counter()
asr_model = Whisper(model_path(LIFEMATE_DIR, CFG["asr_model"]))  # PhoWhisper-large fine-tune, fp16
t_asr = time.perf_counter() - t_load
model = HFBackend(CFG["llm"], model_path(LIFEMATE_DIR, CFG["adapter"]),  # Qwen3.5-4B-text-only NF4 + LoRA
                  max_tokens=MAX_TOKENS, load_4bit=CFG["load_4bit"])
t_llm = time.perf_counter() - t_load - t_asr
print(f"load model: {int((time.perf_counter() - t_load) * 1000)} ms (ASR {int(t_asr * 1000)} ms, LLM {int(t_llm * 1000)} ms)")

## Bước 3: đọc các test case

In [ ]:
from lifemate.data import load_split, zaic_tools
from lifemate.submit import Runner

ROOT = DATA_ROOT
SPLIT = find_split(ROOT, os.environ.get("SPLIT"))  # thư mục con có <tên>_tools_catalog.json: private_test
ToolEnv = zaic_tools(SPLIT, ROOT).ToolEnv  # zaic_tools của split trong /data, không dùng bản trong image
test_cases = load_split(SPLIT, ROOT)  # từng dòng của ROOT/SPLIT/SPLIT.jsonl
if os.environ.get("IDS"):  # chỉ khi thử: một phần các mẫu
    keep = set(Path(os.environ["IDS"]).read_text().split())
    test_cases = [s for s in test_cases if s["id"] in keep]
runner = Runner(SPLIT, ROOT, asr_model, model, time_budget=CFG["time_budget"], search_guard=CFG["search_guard"])
print(f"{SPLIT}: {len(test_cases)} test case")

Warm-up, không tính giờ (giống `predict.sh`).

In [ ]:
# ASR giải mã một giây im lặng, LLM sinh vài token với prompt dựng từ context của mẫu đầu (không audio, không
# ToolEnv), để mẫu đầu tiên không phải trả chi phí khởi động GPU.
if CFG["warmup"] and test_cases:
    print(f"warm-up: {runner.warmup(test_cases[0]) * 1000:.0f} ms")

## Bước 4: dự đoán từng mẫu, ghi kết quả và thời gian

In [ ]:
all_result, traces = [], []
for item in test_cases:  # lần lượt từng mẫu
    t1 = time.perf_counter()
    env = ToolEnv(SPLIT, item["id"], root=str(ROOT))  # đúng một ToolEnv cho mỗi mẫu
    trace = runner.run(item, env, t1)  # đọc wav -> ASR -> LLM -> env.call(...) -> LLM -> ...
    tool_calls, state_hash = env.get_tool_calls(), env.get_state_hash()
    ms = int((time.perf_counter() - t1) * 1000)
    all_result.append({"id": item["id"], "tool_calls": tool_calls, "state_hash": state_hash, "time": ms})
    traces.append(trace)
    if len(all_result) % 50 == 0 or len(all_result) == len(test_cases):
        print(f"{len(all_result)}/{len(test_cases)} mẫu, mẫu vừa xong {ms} ms", flush=True)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULT_DIR / "jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("đã ghi", RESULT_DIR / "jupyter_submission.jsonl")

## Tóm tắt

In [ ]:
from collections import Counter

from lifemate.runinfo import percentile

ms = [r["time"] for r in all_result]
print(f"predict: {len(ms)} mẫu, {sum(ms)} ms, trung bình {sum(ms) / max(len(ms), 1):.1f} ms/mẫu")
print(f"p50 {percentile(ms, .5):.0f} ms, p95 {percentile(ms, .95):.0f} ms, max {max(ms, default=0)} ms, "
      f"trên {CFG['time_budget']:g} s: {sum(t > CFG['time_budget'] * 1000 for t in ms)} mẫu")
print("lý do dừng:", dict(Counter(t["stop"] for t in traces)))
assert [r["id"] for r in all_result] == [s["id"] for s in test_cases]
assert all(type(r["time"]) is int and r["time"] >= 0 for r in all_result)
if os.environ.get("WORK"):  # chỉ khi thử: trace từng mẫu (văn bản ASR, các bước của agent) để so với predict.sh
    from lifemate.data import write_jsonl
    Path(os.environ["WORK"]).mkdir(parents=True, exist_ok=True)
    write_jsonl(Path(os.environ["WORK"]) / "jupyter_trace.jsonl", traces)